# Choosing the Sample Size: the Margin Formula, Backwards

The Statistopia survey of 49 adults delivered a margin of error of 7 cm — and suppose that's just not good enough: you want 3 cm. Then 49 people is suddenly too small a sample, and the question becomes *what is the smallest sample size that achieves the desired margin?* The answer is the margin-of-error formula run in reverse: solve $\text{MOE} \ge z_{1-\alpha/2}\,\sigma/\sqrt{n}$ for $n$ and get $n \ge (z_{1-\alpha/2}\,\sigma/\text{MOE})^2$ — here $(1.96 \times 25/3)^2 = 266.78$, so **267 adults**. It is based on the sixth video of Lesson 7 (Confidence Intervals).

## 1. Introduction

So far the sample size was a given and the margin of error came out; in practice it's usually the other way around — you know how precise the estimate must be, and the sample size is the thing you get to (and must pay to) choose. This notebook inverts the formula, works the video's example, and explores what the inversion costs.

After completing this notebook you will be able to:

- Turn the margin-of-error equation into an inequality and solve it for $n$.
- Apply the sample-size formula $n \ge (z_{1-\alpha/2}\,\sigma/\text{MOE})^2$ and explain why the result is rounded **up**.
- Reproduce the video's answer: a 3 cm margin at 95% with $\sigma = 25$ needs 267 people.
- Quantify the cost of precision (halving the margin quadruples $n$) and of confidence (a bigger $z$, squared).
- Handle the practical wrinkles: fixed budgets, and planning with a guessed $\sigma$.

## 2. Theory

### The setup, recalled

From the last notebook: 6,000 adults, a sample of $n = 49$, $\bar{x} = 170$ cm, $\sigma = 25$ cm, 95% confidence. The margin came out as $1.96 \times 25/\sqrt{49} = 7$ cm and the interval as $[163, 177]$.

Now suppose 7 cm is too much — you want a more exact estimate, say a margin of **3 cm**. With $n = 49$ that's impossible; you need a bigger sample. How much bigger, at minimum?

### The same math, backwards

Everything in the margin formula is known except $n$:

$$
\text{MOE} \;=\; z_{1-\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}
$$

One refinement before solving: a margin *smaller* than 3 would be even better — 3 is the **maximum** margin you're willing to accept. So replace equality with an inequality:

$$
3 \;\ge\; z_{1-\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}
$$

Substitute the knowns and solve for $n$ (multiply both sides by $\sqrt{n}$, divide by 3, square):

$$
3 \ge \frac{1.96 \times 25}{\sqrt{n}}
\quad\Longrightarrow\quad
\sqrt{n} \ge \frac{1.96 \times 25}{3}
\quad\Longrightarrow\quad
n \ge \left(\frac{1.96 \times 25}{3}\right)^{\!2} = 266.78
$$

Since we're talking about people, round to the next whole person: $n = 267$. A sample of **at least 267 adults** delivers a margin of error of at most 3 cm. (Round *up*, always: 266 people would leave the margin a hair above the target.)

### The general formula

$$
\boxed{\;n \;\ge\; \left(\frac{z_{1-\alpha/2}\cdot\sigma}{\text{MOE}}\right)^{\!2}\;}
$$

Plug in any desired margin of error and out comes the required sample size. Note what sits inside the square: precision ($1/\text{MOE}$) and confidence ($z$) both get squared, so each is quadratically expensive — and $\sigma$ does too, so noisier populations need quadratically more data for the same precision.

## 3. Worked Examples

### Example 1: the video's numbers, checked forward

$n = 267$: margin $= 1.96 \times 25/\sqrt{267} = 2.999$ cm ✓ (just under 3).
$n = 266$: margin $= 1.96 \times 25/\sqrt{266} = 3.004$ cm ✗ (just over) — this is why the ceiling, not the nearest integer.

### Example 2: the price list

Same population ($\sigma = 25$), same 95% confidence, different targets:

| desired MOE | required $n$ |
|---|---|
| 7 cm | $(49/7)^2 = 49$ — the original survey |
| 3 cm | $266.78 \to 267$ |
| 1 cm | $(49/1)^2 = 2401$ |

Cutting the margin from 7 to 1 — a factor of 7 — multiplies the sample by $7^2 = 49$. Precision follows an inverse-square price.

### Example 3: demanding more confidence too

Keep MOE $= 3$ but raise the level to 99% ($z = 2.576$):

$$
n \ge \left(\frac{2.576 \times 25}{3}\right)^2 = 460.7 \;\to\; 461
$$

Almost double the 267 — the $z$ inside the square means confidence upgrades are paid for quadratically as well.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(31)

def z_score(p):
    """Critical value z_p: the point with probability p to its left."""
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

def margin_of_error(sigma, n, level=0.95):
    return z_score(1 - (1 - level) / 2) * sigma / np.sqrt(n)

def required_sample_size(moe, sigma, level=0.95):
    """Smallest n whose margin of error is at most moe."""
    z = z_score(1 - (1 - level) / 2)
    return int(np.ceil((z * sigma / moe) ** 2))

### The video's calculation, and the ceiling check

In [ ]:
SIGMA = 25.0

exact = (z_score(0.975) * SIGMA / 3) ** 2
n_needed = required_sample_size(moe=3, sigma=SIGMA)
print(f"(1.96 * 25 / 3)^2 = {exact:.2f}  ->  n = {n_needed}")
print(f"margin at n = {n_needed}: {margin_of_error(SIGMA, n_needed):.4f} cm  (<= 3, good)")
print(f"margin at n = {n_needed - 1}: {margin_of_error(SIGMA, n_needed - 1):.4f} cm "
      f"(> 3 -- one person short)")

### The price list, computed — precision and confidence both squared

In [ ]:
print("95% level, sigma = 25:")
for moe in (7, 3, 1):
    print(f"  MOE <= {moe} cm -> n = {required_sample_size(moe, SIGMA):>5d}")

print("\nMOE <= 3 cm at higher levels:")
for level in (0.90, 0.95, 0.99):
    print(f"  {level:.0%} -> n = {required_sample_size(3, SIGMA, level):>5d}")

### Does 267 keep both promises? Simulate

Draw 10,000 surveys of 267 islanders, build the $\pm 2.999$ cm intervals, and check that ~95% capture the true mean — precision *and* confidence, delivered together.

In [ ]:
MU_TRUE = 170.0
n = required_sample_size(3, SIGMA)
m = margin_of_error(SIGMA, n)

x_bars = rng.normal(MU_TRUE, SIGMA / np.sqrt(n), size=10_000)
coverage = (np.abs(x_bars - MU_TRUE) <= m).mean()
print(f"n = {n}: margin = {m:.3f} cm (target 3), coverage = {coverage:.1%} (target 95%)")

## 5. Visualizations

### Reading the margin curve backwards

The same margin-vs-$n$ curve as two notebooks ago — but now entered from the *vertical* axis: pick the margin you want, slide across to the curve, drop down to read the $n$ it demands.

In [ ]:
n_axis = np.arange(10, 500)
margins = margin_of_error(SIGMA, n_axis)

plt.figure(figsize=(9.5, 5))
plt.plot(n_axis, margins, color="tab:blue", linewidth=2,
         label=r"margin $= 1.96 \cdot 25/\sqrt{n}$")
for moe, n_req, color in [(7, 49, "tab:green"), (3, 267, "tab:red")]:
    plt.plot([0, n_req], [moe, moe], color=color, linestyle="--", linewidth=1.3)
    plt.plot([n_req, n_req], [0, moe], color=color, linestyle="--", linewidth=1.3)
    plt.plot(n_req, moe, "o", color=color, markersize=6,
             label=f"MOE = {moe} cm -> n = {n_req}")
plt.xlim(0, 500)
plt.ylim(0, 12)
plt.xlabel("sample size n")
plt.ylabel("margin of error (cm)")
plt.title("Choose the margin, read off the sample size")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The inverse-square price of precision

In [ ]:
moe_axis = np.linspace(0.8, 8, 300)
n_curve = (z_score(0.975) * SIGMA / moe_axis) ** 2

plt.figure(figsize=(9.5, 4.6))
plt.plot(moe_axis, n_curve, color="tab:red", linewidth=2)
for moe in (7, 3, 1):
    n_pt = required_sample_size(moe, SIGMA)
    plt.plot(moe, n_pt, "ko", markersize=5)
    plt.annotate(f"{moe} cm: n = {n_pt:,}", xy=(moe, n_pt),
                 xytext=(8, 8), textcoords="offset points", fontsize=9)
plt.xlabel("desired margin of error (cm)")
plt.ylabel("required sample size n")
plt.title(r"$n = (1.96\,\sigma/\mathrm{MOE})^2$: halve the margin, quadruple the sample")
plt.grid(True, alpha=0.3)
plt.show()

### The whole story on one page

Everything above in a single sheet: the margin curve read backwards, the n = 267 calculation, the price list, and the inverse-square cost of precision.

<img src="img/05_choosing_sample_size.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** How many islanders are needed for a 5 cm margin at 95% ($\sigma = 25$)? And for 2 cm?

**Exercise 2.** Without computing: the required $n$ for some target comes out to 108.02. A colleague suggests $n = 108$ ("it's basically there"). What margin does $n = 108$ actually deliver relative to the target, and what's the safe answer?

### Intermediate

**Exercise 3.** Derive the sample-size formula with SymPy: solve the boundary equation $\text{MOE} = z\sigma/\sqrt{n}$ for $n$, then argue in one sentence why the inequality version is $n \ge (z\sigma/\text{MOE})^2$ rather than $\le$.

**Exercise 4.** *Fixed budget.* You can only afford to survey 150 islanders. (a) What 95% margin do you get? (b) What confidence level would you have to settle for to claim a 3 cm margin with those 150 people?

### Advanced

**Exercise 5.** *One person short.* Simulate 20,000 surveys each at $n = 266$ and $n = 267$ and report the margins and coverages. Confirm both cover at 95% — the difference is purely that 266 breaks the *width* promise, not the confidence one. Which promise does the ceiling protect?

**Exercise 6.** *Planning with a guessed $\sigma$.* In reality $\sigma$ is estimated before the survey. If you plan $n$ for a 3 cm margin using guess $\sigma_g = 25$ but the true $\sigma$ is 10% larger, what margin do you actually get? Plot the delivered margin against the true $\sigma$ for the planned $n = 267$, and state the practical rule this suggests.

## 7. Solutions

In [ ]:
# Exercise 1: two more targets
for moe in (5, 2):
    print(f"MOE <= {moe} cm -> n >= {(z_score(0.975) * SIGMA / moe) ** 2:.2f} "
          f"-> n = {required_sample_size(moe, SIGMA)}")

In [ ]:
# Exercise 2: why 108.02 -> 109
print("n = 108 leaves the margin just ABOVE the target (the inequality n >= 108.02")
print("fails), so the promised precision is not met -- barely, but a promise is a")
print("promise. The safe answer is always the ceiling: n = 109.")

In [ ]:
# Exercise 3: the derivation with sympy
n_sym, z_sym, sigma_sym, moe_sym = sp.symbols("n z sigma MOE", positive=True)
boundary = sp.solve(sp.Eq(moe_sym, z_sym * sigma_sym / sp.sqrt(n_sym)), n_sym)[0]
print("Exercise 3: the boundary case MOE = z*sigma/sqrt(n) solves to n =", boundary)
print("The margin SHRINKS as n grows, so any n at or beyond the boundary keeps")
print("MOE at or below the target: the requirement is n >= (z*sigma/MOE)^2.")

In [ ]:
# Exercise 4: fixed budget of 150
m_150 = margin_of_error(SIGMA, 150)
print(f"(a) n = 150 -> 95% margin = {m_150:.2f} cm")

z_150 = 3 * np.sqrt(150) / SIGMA               # z that makes the margin exactly 3
level_150 = float(sp.erf(z_150 / sp.sqrt(2)))
print(f"(b) 3 cm at n = 150 needs z = {z_150:.3f} -> confidence level = {level_150:.1%}")
print("    A 3 cm claim from 150 people is only an ~86% promise -- below the 90%")
print("    floor practitioners accept, so the honest move is more people, not less confidence.")

In [ ]:
# Exercise 5: n = 266 vs 267
for n_test in (266, 267):
    m_test = margin_of_error(SIGMA, n_test)
    xb = rng.normal(MU_TRUE, SIGMA / np.sqrt(n_test), size=20_000)
    cov = (np.abs(xb - MU_TRUE) <= m_test).mean()
    ok = "meets" if m_test <= 3 else "MISSES"
    print(f"n = {n_test}: margin = {m_test:.4f} cm ({ok} the 3 cm target), "
          f"coverage = {cov:.1%}")
print("\nCoverage is 95% either way -- the recipe always keeps its confidence")
print("promise. The ceiling protects the WIDTH promise: the margin you advertised.")

In [ ]:
# Exercise 6: sensitivity to the sigma guess
n_planned = 267
sigma_true_axis = np.linspace(20, 32.5, 200)
delivered = z_score(0.975) * sigma_true_axis / np.sqrt(n_planned)

plt.figure(figsize=(9, 4.2))
plt.plot(sigma_true_axis, delivered, color="tab:blue", linewidth=2)
plt.axhline(3, color="black", linestyle=":", label="3 cm target")
plt.axvline(25, color="gray", linestyle="--", linewidth=1, label=r"planning guess $\sigma_g = 25$")
sigma_110 = 27.5
plt.plot(sigma_110, z_score(0.975) * sigma_110 / np.sqrt(n_planned), "ro",
         label=f"true sigma 10% larger: margin = "
               f"{z_score(0.975) * sigma_110 / np.sqrt(n_planned):.2f} cm")
plt.xlabel(r"true population $\sigma$ (cm)")
plt.ylabel("delivered margin at n = 267 (cm)")
plt.title(r"Exercise 6: an optimistic $\sigma$ guess quietly breaks the width promise")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

print(f"sigma 10% larger -> margin {3 * 1.1:.1f} cm (10% larger too, since margin")
print("is linear in sigma). Practical rule: plan with a conservative (generous)")
print("sigma guess -- overshooting n costs money, undershooting breaks the promise.")

## 8. Conclusion

- The margin formula runs backwards: from $\text{MOE} \ge z_{1-\alpha/2}\,\sigma/\sqrt{n}$, the smallest adequate sample is $n \ge (z_{1-\alpha/2}\,\sigma/\text{MOE})^2$ — and you round **up**, because one person short breaks the width promise (never the coverage one).
- The video's numbers: shrinking Statistopia's margin from 7 cm to 3 cm raises the survey from 49 to $(1.96 \times 25/3)^2 = 266.78 \to 267$ adults.
- Everything in the formula is squared: 7× the precision costs 49× the sample; 99% instead of 95% confidence at the same margin nearly doubles it; a noisier population scales it by $\sigma^2$.
- Practical wrinkles: with a fixed budget you can only trade margin against confidence (150 people = a 4 cm margin at 95%, or 3 cm at a weak 86%), and since planning uses a *guessed* $\sigma$, guess generously — the delivered margin scales linearly with the true $\sigma$.

Next: a short but important clarification -- what the 95% is and isn't a probability *of* -- and then the last assumption still standing over the whole lesson, the known $\sigma$.